# v38 — 基于原始 33.89 baseline 的最小性能补丁

本 notebook 以 `baseline_LB33.89_failed-in-aimo.ipynb` 为母版：**保留原有 9 个 cell、原 metadata、模型路径、竞赛输入、4×L4 并行、128 条 task-time 训练增强、16 条推理增强、LoRA 参数、解码阈值、候选聚合和 submission schema。**

仅在原 `arc_solver.py` 的两个 logits 热点中做定点替换：

1. `turbo_dfs`：仍对同一组 12 个 ARC token 使用同样的 NLL、阈值、候选排序与 DFS；仅把 full-vocabulary logits 的归一化留在 GPU，并只把 12 个 token 的结果传回 CPU。
2. `calc_scores`：仍使用同一 teacher-forced NLL 作为重排序分数；仅在 GPU 上 gather 目标 token 并关闭未被消费的 KV cache。

没有缩短或延长原 `global_end_time = now + 12h − 10min`，也没有改动 `starter.py`、结果写入、队列策略或提交逻辑。该版本的目的仅是减少 CPU↔GPU logits 传输与多余 KV cache 写入，从而在**不增加时间预算**的前提下提高已完成任务覆盖率。

In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile gen_boost.py
"""Generation-side levers for the AGI-2 perfpatch kernel, with the decision rule.

    python3 gen_boost.py --decide /tmp/evalscan/select_verdict.json [--slack 0.4]
    <venv with torch>/python gen_boost.py --selftest       # CPU, fake LM, real arc_loader / turbo_dfs / calc_scores

`decode_task` replaces the decode block of arc_solver.worker() (the part after
TTT). With every lever off it writes byte-identical pickles to production --
that is asserted in the selftest by running the ORIGINAL block, extracted from
arc_solver.py, against the same fake model.

Levers (all gated by the per-task time cap and the global end_time):
  g1  greedy fallback. The DFS keeps a beam only while its cumulative
      probability stays above 0.2; a 30x30 answer (~930 tokens) needs
      p > 0.9983 per token, so one uncertain cell empties the view and the
      input ships the [[0]] placeholder. Greedy is one forward pass per token.
        "empty_input": only for test inputs with NO candidate after the normal
                       pass -- cannot change any input that already had candidates
        "empty_view":  for every empty view -- adds 1-vote candidates to inputs
                       that have others; needs its own A/B
  g3  16 extra views (new colour permutations / example orders) for inputs
      whose best grid is backed by < min_agree views
  g2  one more DFS over the base views with the cut loosened to p > 0.05,
      keeping only beams the first pass could not have produced
  faircap  per-task cap from the time actually left instead of a fixed 1200 s
  probe_gold  measurement only, never in a competition rerun: teacher-forced
      NLL of the GOLD grid under the TTT'd model on the 8 aug-scoring views,
      appended to <probe_dir>/gold_nll_rank<r>.jsonl. Tells, per missed input,
      whether the answer was just outside the decoder's reach or nowhere near
      (read by gen_autopsy.py --gold-nll). Costs two scoring batches per input.

Measured locally and therefore NOT implemented: raising max_seq_length (3/172
eval inputs lose a train example at 8192, none extraction-type) and extra TTT
epochs (production already ends each task at training_loss ~1e-4).
"""
import bz2
import json
import math
import os
import pickle
import sys
import time
from collections import defaultdict

import numpy as np

try:
    import torch
except ImportError:                                    # --decide works without torch
    torch = None

LEVERS_OFF = {"g1": "off", "g3": False, "g2": False, "faircap": False, "probe_gold": False}
GOLD_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
BASE_CAP = 1200.0
LOOSE_P = 0.05
MIN_AGREE = 3


# --------------------------------------------------------------------- decode
def greedy_decode(model, prefix_tokens, max_new_tokens, end_time, arc_token_ids, eos_id, pad_id):
    """Arg-max decode for a batch of equal-length prefixes, restricted to the
    ARC vocabulary. Same return shape as inference_turbo_dfs:
    [(batch_id, [(nll, tokens)])], only rows that reached EOS."""
    with torch.no_grad():
        input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
        out = model(input_ids=input_ids, return_dict=True, use_cache=True)
        n, pos = input_ids.size(0), input_ids.size(1)
        ids = arc_token_ids(out.logits.device)
        seqs = [[] for _ in range(n)]
        nll = [0.0] * n
        alive = [True] * n
        done = [False] * n
        logits, cache = out.logits[:, -1], out.past_key_values
        for _ in range(max_new_tokens):
            lf = logits.float()
            lp = lf.index_select(-1, ids) - torch.logsumexp(lf, dim=-1, keepdim=True)
            best = lp.argmax(dim=-1)
            best_lp = lp.gather(-1, best.view(-1, 1)).view(-1).cpu().tolist()
            toks = ids[best].cpu().tolist()
            feed = []
            for i in range(n):
                if not alive[i]:
                    feed.append(pad_id)
                    continue
                nll[i] -= best_lp[i]
                seqs[i].append(toks[i])
                if toks[i] == eos_id:
                    alive[i], done[i] = False, True
                    feed.append(pad_id)
                else:
                    feed.append(toks[i])
            if not any(alive) or time.time() > end_time:
                break
            out = model(
                input_ids=torch.tensor(feed, device=model.device, dtype=torch.long).view(-1, 1),
                position_ids=torch.full((n, 1), pos, device=model.device),
                past_key_values=cache,
                return_dict=True,
                use_cache=True,
            )
            logits, cache = out.logits[:, -1], out.past_key_values
            pos += 1
    return [(i, [(nll[i], seqs[i])]) for i in range(n) if done[i]]


def make_batches(keys):
    """Production batching: per test input, rotation-family batches for every
    test first, then transpose-family batches (equal prompt lengths per batch)."""
    by_test = defaultdict(list)
    for subkey in sorted(keys):
        by_test[subkey.split(".")[0].split("_")[1]].append(subkey)
    batches = []
    for offsets in (([0, 4], [2, 6]), ([8, 12], [10, 14])):
        for _, subkeys in by_test.items():
            for pair in offsets:
                batch = []
                for off in pair:
                    batch.extend(subkeys[off:off + 2])
                batches.append(batch)
    return batches


def agreement(decoded_by_view):
    """{view: [sample, ...]} for one test input -> views agreeing on the best-supported grid."""
    votes = defaultdict(int)
    for samples in decoded_by_view.values():
        for h in {tuple(map(tuple, s["solution"])) for s in samples}:
            votes[h] += 1
    return max(votes.values()) if votes else 0


def fair_share_seconds(end_time, tasks_left, n_workers=4, floor=600.0, ceil=2400.0, now=None):
    """Time this task may use if the remaining budget were split evenly over
    the tasks this worker still has to do (itself included)."""
    remaining = max(0.0, end_time - (time.time() if now is None else now))
    mine = math.ceil(max(tasks_left, 0) / n_workers) + 1
    return min(ceil, max(floor, remaining / mine))


class WaitQueue:
    """Queue view for a second pass fed by workers that are still in their first pass.
    empty() blocks until an item arrives or every producer has reported done, so the kernel's
    `while not queue.empty(): key = queue.get()` loop works unchanged. `done` is a queue the
    producers put one token into when they finish; qsize() adds the 4 sentinels the kernel's
    tasks_left arithmetic subtracts."""

    def __init__(self, q, done, n_workers=4, poll=2.0, sleep=time.sleep):
        self.q, self.done, self.n, self.poll, self.sleep, self.buf = q, done, n_workers, poll, sleep, []

    def empty(self):
        while not self.buf:
            try:
                self.buf.append(self.q.get_nowait())
            except Exception:                                  # noqa: BLE001  (queue.Empty, also via a Manager proxy)
                if self.done.qsize() >= self.n and self.q.qsize() == 0:
                    return True
                self.sleep(self.poll)
        return False

    def get(self):
        return None if self.empty() else self.buf.pop(0)

    def qsize(self):
        return self.q.qsize() + len(self.buf) + 4


def unsettled(res):
    """True when some test input of the task has no grid backed by MIN_AGREE views (or no candidate at all)."""
    return any(agreement(res["by_input"].get(bk, {})) < MIN_AGREE for bk in res["inputs"])


def decode_task(ctx, puzzle_ds_multi, start_time, end_time, levers=None, tasks_left=0):
    """Decode every test input of one task. ctx carries what worker() already
    has: model, tokenizer, formatter, ArcDataset, inference_turbo_dfs,
    calc_scores, arc_token_ids, EOS_ID, PAD_ID, max_new_tokens, max_score,
    max_seq_length, dir_outputs, rank. Optional: file_tag (appended to every
    file name, e.g. ".runm2" for a second model's candidates) and cap_ceil
    (upper bound of the fair-share cap)."""
    lv = dict(LEVERS_OFF)
    lv.update(levers or {})
    model, tokenizer, formatter = ctx["model"], ctx["tokenizer"], ctx["formatter"]
    rank, max_new, max_score = ctx["rank"], ctx["max_new_tokens"], ctx["max_score"]
    in_len = ctx["max_seq_length"] - max_new
    task_cap = fair_share_seconds(end_time, tasks_left, ceil=ctx.get("cap_ceil", 2400.0)) if lv["faircap"] else BASE_CAP
    file_tag = ctx.get("file_tag", "")
    stats = defaultdict(int)
    by_input = defaultdict(dict)                 # base key -> {file name: [sample]}
    known_scores = {}

    def out_of_time():
        return time.time() - start_time > task_cap or time.time() > end_time

    def aug_scores(bk, solution):
        """NLL of `solution` for input bk under the 8 augmented scoring views (production recipe)."""
        aug_dataset = ctx["ArcDataset"](
            keys=[bk],
            queries={bk: puzzle_ds_multi.queries.get(bk)},
            replies={bk: [np.asarray(solution).tolist()]},
        )
        aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
        aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=in_len)
        aug_queries, aug_answers = [], []
        for augmented_sample in aug_dataset.as_list(formatter):
            aug_queries.append(augmented_sample["input"])
            aug_answers.append(augmented_sample["reply"])
        return (ctx["calc_scores"](aug_queries[:4], aug_answers[:4], tokenizer, model)
                + ctx["calc_scores"](aug_queries[4:], aug_answers[4:], tokenizer, model))

    def score_and_store(ds, subkey, scored_beams, suffix, keep=None):
        bk = subkey.split(".")[0]
        decoded_result = []
        for beam_score, tokens in scored_beams:
            if keep is not None and not keep(beam_score):
                continue
            array = formatter.convert_tokens_to_array(tokens)
            if array is None:
                continue
            solution = ds.invert_mod(array, subkey, inv_perm=True)
            grid_id = (bk, tuple(map(tuple, solution)))
            if grid_id in known_scores:
                augmented_scores = known_scores[grid_id]
            else:
                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                augmented_scores = aug_scores(bk, solution)
                known_scores[grid_id] = augmented_scores
            decoded_result.append({"beam_score": beam_score, "score_aug": augmented_scores, "solution": solution})
        if decoded_result:
            name = subkey + file_tag + suffix
            with bz2.BZ2File(os.path.join(ctx["dir_outputs"], name), "w") as f:
                pickle.dump(decoded_result, f)
            by_input[bk][name] = decoded_result
        return len(decoded_result)

    def run_pass(ds, batches, mode, suffix="", cut=None, keep=None, label=""):
        """mode: 'dfs' | 'greedy' | 'dfs+greedy' (greedy for views the DFS left empty)."""
        for subkeys in batches:
            if not subkeys:
                continue
            if out_of_time():
                print(f"[Rank {rank}] timeout after {time.time() - start_time:.1f}s for puzzle {ctx.get('key', '')}{label}")
                return False
            print(f"[Rank {rank}] decoding {subkeys}{label}")
            tokens = [tokenizer.encode(ds.get(subkey, formatter)["input"]) for subkey in subkeys]
            result = []
            if mode != "greedy":
                result = ctx["inference_turbo_dfs"](model, tokens, max_new, max_score if cut is None else cut, end_time)
            got = set()
            for subkey_id, scored_beams in result:
                if score_and_store(ds, subkeys[subkey_id], scored_beams, suffix, keep):
                    got.add(subkey_id)
            if mode != "dfs":
                empty = [i for i in range(len(subkeys)) if i not in got]
                if empty and time.time() < end_time:
                    extra = greedy_decode(model, [tokens[i] for i in empty], max_new, end_time,
                                          ctx["arc_token_ids"], ctx["EOS_ID"], ctx["PAD_ID"])
                    for j, beams in extra:
                        stats["g1_grids"] += score_and_store(ds, subkeys[empty[j]], beams, suffix + ".rung")
        return True

    eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
    eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=in_len)
    base_batches = make_batches(eval_ds.keys)
    input_keys = sorted({k.split(".")[0] for k in eval_ds.keys})

    with torch.inference_mode():
        # pass 1 -- production behaviour (plus per-view greedy if asked for)
        run_pass(eval_ds, base_batches, "dfs+greedy" if lv["g1"] == "empty_view" else "dfs")

        # G1 empty_input -- inputs that would otherwise ship the placeholder
        if lv["g1"] == "empty_input":
            for bk in input_keys:
                if not by_input[bk] and not out_of_time():
                    stats["g1_inputs"] += 1
                    mine = [[s for s in b if s.split(".")[0] == bk] for b in base_batches]
                    run_pass(eval_ds, mine, "greedy", label=" [g1]")

        # G3 -- more views where there is no consensus
        if lv["g3"]:
            for bk in input_keys:
                if agreement(by_input[bk]) >= MIN_AGREE or out_of_time():
                    continue
                stats["g3_inputs"] += 1
                extra_ds = puzzle_ds_multi.change_keys([bk]).augment(n=2, seed=3)
                extra_ds = extra_ds.cut_to_len(formatter=formatter, name="input", max_len=in_len)
                run_pass(extra_ds, make_batches(extra_ds.keys), "dfs", suffix=".run2", label=" [g3]")

        # G2 -- looser cut, only beams the first pass could not have produced
        if lv["g2"]:
            loose = -math.log(LOOSE_P)
            for bk in input_keys:
                if agreement(by_input[bk]) >= MIN_AGREE or out_of_time():
                    continue
                stats["g2_inputs"] += 1
                mine = [[s for s in b if s.split(".")[0] == bk] for b in base_batches]
                run_pass(eval_ds, mine, "dfs", suffix=".run3", cut=loose,
                         keep=lambda nll: nll >= max_score, label=" [g2]")

        # probe_gold -- measurement only; impossible in a rerun (no solutions file, and guarded anyway)
        if lv["probe_gold"] and not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
            gold_path = ctx.get("gold_path", GOLD_PATH)
            if os.path.exists(gold_path):
                with open(gold_path) as f:
                    gold_all = json.load(f)
                lines = []
                for bk in input_keys:
                    task, idx = bk.split("_")
                    if task not in gold_all or time.time() > end_time:
                        continue
                    gold = np.asarray(gold_all[task][int(idx)])
                    gid = (bk, tuple(map(tuple, gold)))
                    nll = known_scores[gid] if gid in known_scores else aug_scores(bk, gold)
                    lines.append(json.dumps({"bk": bk, "gold_aug_nll": [float(x) for x in nll],
                                             "n_tok": int(gold.shape[0] * (gold.shape[1] + 1)),
                                             "generated": gid in known_scores}))
                    stats["probe_inputs"] += 1
                if lines:
                    with open(os.path.join(ctx.get("probe_dir", "/kaggle/working"), f"gold_nll_rank{rank}.jsonl"), "a") as f:
                        f.write("\n".join(lines) + "\n")

    if any(v != LEVERS_OFF[k] for k, v in lv.items()):
        print(f"[Rank {rank}] gen_boost levers={lv} cap={task_cap:.0f}s stats={dict(stats)}")
    return {"stats": dict(stats), "by_input": by_input, "task_cap": task_cap, "inputs": input_keys}


# ------------------------------------------------------------------- decision
def union_pairs(decoded_results, tag=".runm2"):
    """Grids that BOTH models generated for the same test input -> [(mean aug NLL under A, under B, base key)].
    Each model scored the grid itself, with its own per-task adapter, so these pairs show how the two NLL
    scales relate on grids each model 'owns' equally."""
    pairs = []
    for bk, guesses in decoded_results.items():
        by = {}
        for name, g in guesses.items():
            h = tuple(map(tuple, g["solution"]))
            by.setdefault(h, ([], []))[1 if tag in name else 0].append(float(np.mean(g["score_aug"])))
        for a, b in by.values():
            if a and b:
                pairs.append((float(np.mean(a)), float(np.mean(b)), bk))
    return pairs


def calibrate_union(decoded_results, tag=".runm2", mode="offset", min_pairs=8):
    """Put model B's aug scores on model A's scale before selection. In place; returns what was done.

    kgmon ranks a grid by  votes - mean aug NLL, and every sample carries the NLL of the model that
    generated it. If model B's NLLs sit systematically above (below) A's, B-only grids are under- (over-)
    ranked against A-only grids of the same quality. The relation is estimated on the grids both models
    generated (union_pairs, pooled over all inputs of the run):
        offset: B' = B - median(B - A)        ratio: B' = B / median(B / A)
    Fewer than min_pairs shared grids -> nothing is changed (no evidence, no correction).
    Samples of model A are never touched, so tasks without pass-2 output select exactly as before."""
    pairs = union_pairs(decoded_results, tag)
    info = {"pairs": len(pairs), "mode": mode, "applied": False, "offset": 0.0, "ratio": 1.0, "b_samples": 0}
    if pairs:
        d = [b - a for a, b, _ in pairs]
        r = [b / a for a, b, _ in pairs if a > 1e-3]
        info["offset"] = float(np.median(d))
        info["offset_iqr"] = [float(np.percentile(d, 25)), float(np.percentile(d, 75))]
        info["ratio"] = float(np.median(r)) if r else 1.0
    if len(pairs) < min_pairs or mode not in ("offset", "ratio"):
        return info
    off, ratio = info["offset"], max(info["ratio"], 1e-3)
    for guesses in decoded_results.values():
        for name in list(guesses):
            if tag in name:
                g = guesses[name]
                new = [float(x) - off for x in g["score_aug"]] if mode == "offset" else [float(x) / ratio for x in g["score_aug"]]
                guesses[name] = dict(g, score_aug=new, score_aug_raw=list(g["score_aug"]))
                info["b_samples"] += 1
    info["applied"] = True
    return info


def decide(diag, slack=None):
    """gen_diagnostics dict (select_verdict.json['diagnostics']) -> levers + reasons.
    slack = fraction of the kernel budget the evalscan run left unused
    (1 - sum of per-task TOTAL / (4 workers x budget)); None = unknown."""
    n = max(diag["n_inputs"], 1)
    bins = diag["gold_bins"]
    gold = sum(bins.values())
    low = bins["p .3-.5"] + bins["p .2-.3"]
    weak = diag["low_consensus_with_gold"] + diag["low_consensus_without_gold"]
    levers, why = dict(LEVERS_OFF), []

    if diag["zero_cand"] >= 3:
        levers["g1"] = "empty_input"
        why.append(f"g1=empty_input: {diag['zero_cand']}/{n} inputs have no candidate at all "
                   f"(by class {diag['zero_by_class']}); greedy can only replace their placeholder")
    else:
        why.append(f"g1 off: only {diag['zero_cand']} zero-candidate inputs")

    time_ok = slack is None or slack >= 0.2
    if gold >= 8 and low / gold >= 0.25:
        if time_ok:
            levers["g2"] = True
        why.append(f"g2 {'on' if levers['g2'] else 'blocked by time'}: {low}/{gold} generated golds sit at best-view "
                   "p < 0.5 -- the 0.2 cut is censoring a visible tail")
    else:
        why.append(f"g2 off: {low}/{gold} generated golds below p 0.5 -- no sign of censoring at the cut")

    if weak >= 5:
        if time_ok:
            levers["g3"] = True
        why.append(f"g3 {'on' if levers['g3'] else 'blocked by time'}: {weak} inputs have < {MIN_AGREE} agreeing views "
                   f"({diag['low_consensus_with_gold']} with the gold already generated)")
    else:
        why.append(f"g3 off: only {weak} low-consensus inputs")

    if slack is not None and slack >= 0.2 and (levers["g2"] or levers["g3"] or levers["g1"] != "off"):
        levers["faircap"] = True
        why.append(f"faircap on: {slack:.0%} of the budget was unused")
    elif slack is None and (levers["g2"] or levers["g3"]):
        why.append("slack unknown: g2/g3 enabled under the fixed 1200 s cap only -- pass --slack from timing.log")

    cap_share = diag["wrong_only"] / n
    why.append(f"capability: {diag['wrong_only']}/{n} inputs have candidates but never the gold "
               f"({cap_share:.0%}) -- no decode lever reaches these; that share is the model-swap target")
    return {"levers": levers, "why": why, "changed": levers != LEVERS_OFF}


# ------------------------------------------------------------------- selftest
def _extract(solver_src, names):
    """Source of selected top-level defs / assignments from arc_solver.py
    (the module itself imports unsloth and cannot be imported here)."""
    import ast
    tree = ast.parse(solver_src)
    out = []
    for node in tree.body:
        nm = getattr(node, "name", None)
        if nm is None and isinstance(node, ast.Assign):
            nm = node.targets[0].id if isinstance(node.targets[0], ast.Name) else None
        if nm in names:
            seg = ast.get_source_segment(solver_src, node)
            decos = "".join("@" + ast.get_source_segment(solver_src, d) + "\n" for d in getattr(node, "decorator_list", []))
            out.append(decos + seg)
    return "\n\n".join(out)


def selftest():
    import re
    import tempfile
    import types
    here = os.path.dirname(os.path.abspath(__file__))
    prod = os.path.join(os.path.dirname(os.path.dirname(here)), "submit_ag2_perfpatch", "out")
    if "transformers" not in sys.modules:
        try:
            import transformers  # noqa: F401
        except ImportError:
            sys.modules["transformers"] = types.SimpleNamespace(AutoTokenizer=object)
    sys.path.insert(0, prod)
    sys.path.insert(0, here)
    from arc_loader import ArcDataset, QwenFormatter
    import arc_decoder
    import select_v2

    solver_src = open(os.path.join(prod, "arc_solver.py")).read()
    ns = {"torch": torch, "time": time, "defaultdict": defaultdict, "np": np}
    exec(_extract(solver_src, {"ARC_VOCAB", "ARC_TOKENS", "USER_TOKEN_ID", "ASSISTANT_TOKEN_ID", "PAD_ID", "EOS_ID",
                               "_ARC_TOKEN_ID_CACHE", "_arc_token_ids", "turbo_dfs", "inference_turbo_dfs",
                               "calc_scores"}), ns)
    EOS, PAD, V = ns["EOS_ID"], ns["PAD_ID"], 16

    class Tok:                                           # the kernel's WordLevel tokenizer: 1 token per symbol
        pat = re.compile(r"<\|im_start\|>|<\|im_end\|>|user|assistant|\n|\d")
        ids = {"<|im_start|>": 14, "<|im_end|>": 15, "user": 11, "assistant": 12, "\n": 10}

        def encode(self, text):
            return [self.ids[m] if m in self.ids else int(m) for m in self.pat.findall(text)]

        def decode(self, toks):
            inv = {v: k for k, v in self.ids.items()}
            return "".join(inv.get(int(t), str(int(t))) for t in toks)

    tok = Tok()
    fmt = QwenFormatter(tokenizer=tok)

    class FakeLM:
        """Puts probability conf(prompt) on the next token of the registered
        answer for that prompt, the rest on one distractor; uniform off-path.
        past_key_values is the immutable token history, so DFS siblings can
        share a cache exactly as the real KV cache allows."""
        device = torch.device("cpu")

        def __init__(self):
            self.answers, self.conf, self.calls = {}, {}, 0

        def _row(self, hist):
            hist = hist.tolist()
            out = torch.full((len(hist), V), -math.log(V))
            head = max((i for i in range(len(hist) - 1) if hist[i] == 12 and hist[i + 1] == 10), default=None)
            if head is None:
                return out
            key = tuple(hist[:head + 2])
            if key not in self.answers:
                return out
            ans, c = self.answers[key], self.conf.get(key, 0.97)
            for t in range(head + 1, len(hist)):
                gen = hist[head + 2:t + 1]
                if gen == ans[:len(gen)] and len(gen) < len(ans):
                    row = torch.full((V,), -30.0)
                    nxt = ans[len(gen)]
                    row[nxt] = math.log(c)
                    row[(nxt + 1) % 10 if nxt < 10 else 0] = math.log(1 - c)
                    out[t] = row
            return out

        def __call__(self, input_ids, position_ids=None, past_key_values=None, return_dict=True, use_cache=True):
            self.calls += 1
            hist = input_ids if past_key_values is None else torch.cat([past_key_values, input_ids], dim=1)
            full = torch.stack([self._row(h) for h in hist])
            logits = full[:, -input_ids.size(1):]
            return types.SimpleNamespace(logits=logits, past_key_values=hist if use_cache else None)

    def task(seed, n_test=1):
        r = np.random.default_rng(seed)
        f = lambda g: np.flipud(g).tolist()                      # noqa: E731
        mk = lambda: r.integers(0, 10, size=(3, 3)).tolist()     # noqa: E731
        tr = [{"input": (g := mk()), "output": f(g)} for _ in range(3)]
        te = [{"input": mk()} for _ in range(n_test)]
        return {"train": tr, "test": te}, [f(t["input"]) for t in te]

    def register(lm, ds_with_replies, key, conf_of):
        """Teach the fake LM the right answer for every prompt the kernel will
        build for `key`: decode views (seed 2), G3 views (seed 3), aug-score views."""
        multi = ds_with_replies.change_keys([key]).split_multi_replies()
        # same construction (and therefore the same RNG draws) as the kernel for each kind of view
        views = [(2, multi.augment(n=2, seed=2))]                                  # pass 1: all inputs at once
        for bk in multi.keys:
            views.append((3, multi.change_keys([bk]).augment(n=2, seed=3)))        # G3: one input at a time
            sd = hash(bk) % 1024**2
            views.append((sd, multi.change_keys([bk]).augment(seed=sd)))           # aug scoring
        for sd, d in views:
            for sub in d.keys:
                item = d.get(sub, fmt)
                p = tuple(tok.encode(item["input"]))
                lm.answers[p] = tok.encode(item["reply"])
                lm.conf[p] = conf_of(sd, sub)

    def ctx_for(lm, out_dir):
        return dict(model=lm, tokenizer=tok, formatter=fmt, ArcDataset=ArcDataset,
                    inference_turbo_dfs=ns["inference_turbo_dfs"], calc_scores=ns["calc_scores"],
                    arc_token_ids=ns["_arc_token_ids"], EOS_ID=EOS, PAD_ID=PAD,
                    max_new_tokens=fmt.max_new_tokens(), max_score=-np.log(0.2), max_seq_length=8192,
                    dir_outputs=out_dir, rank=0)

    def original_block(lm, multi, out_dir):
        """The production decode block, sliced out of arc_solver.py and run as-is."""
        a = solver_src.index("        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)\n")
        b = solver_src.index("        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2\n        print(f\"[Rank {rank}] allocated {memory_allocated}MB for inference\")")
        block = "\n".join(l[8:] for l in solver_src[a:b].splitlines())
        env = dict(ns, puzzle_ds_multi=multi, formatter=fmt, tokenizer=tok, model=lm, max_seq_length=8192,
                   max_new_tokens=fmt.max_new_tokens(), max_score=-np.log(0.2), start_time=time.time(),
                   end_time=time.time() + 600, rank=0, key="t", dir_outputs=out_dir, ArcDataset=ArcDataset,
                   bz2=bz2, pickle=pickle, os=os)
        exec(compile(block, "original_block", "exec"), env)

    def load(d):
        out = {}
        for f in sorted(os.listdir(d)):
            with bz2.BZ2File(os.path.join(d, f)) as fh:
                out[f] = pickle.load(fh)
        return out

    def same(a, b):
        return a.keys() == b.keys() and all(
            len(a[k]) == len(b[k]) and all(
                x["beam_score"] == y["beam_score"] and x["score_aug"] == y["score_aug"]
                and np.array_equal(x["solution"], y["solution"]) for x, y in zip(a[k], b[k])) for k in a)

    quiet = open(os.devnull, "w")
    real_stdout = sys.stdout

    def run(lm, multi, levers, cap_end=600, tasks_left=0, start=None):
        d = tempfile.mkdtemp()
        sys.stdout = quiet
        try:
            r = decode_task(ctx_for(lm, d), multi, time.time() if start is None else start,
                            time.time() + cap_end, levers, tasks_left)
        finally:
            sys.stdout = real_stdout
        return load(d), r

    queries, replies = {}, {}
    for name, seed, nt in (("aaaa0001", 1, 1), ("aaaa0002", 2, 2), ("aaaa0003", 3, 1), ("aaaa0004", 4, 1), ("aaaa0005", 5, 1)):
        queries[name], replies[name] = task(seed, nt)
    ds_gold = ArcDataset(queries=queries, replies=replies, is_orig=True)
    ds_run = ArcDataset(queries=queries, is_orig=True)             # what the kernel sees: no replies

    def gold_in(files, key, i=0):
        g = np.asarray(replies[key][i])
        return sum(any(np.array_equal(s["solution"], g) for s in v) for k, v in files.items()
                   if k.startswith(f"{key}_{i}."))

    lm = FakeLM()
    L = len(tok.encode(fmt.fmt_reply([replies["aaaa0001"][0]])))                  # 12 tokens for a 3x3 grid
    register(lm, ds_gold, "aaaa0001", lambda sd, sub: 0.97)                         # confident everywhere
    register(lm, ds_gold, "aaaa0002", lambda sd, sub: 0.97)                         # 2 test inputs
    register(lm, ds_gold, "aaaa0003", lambda sd, sub: 0.80)                         # 0.8^12 = 0.07 < 0.2: DFS finds nothing
    register(lm, ds_gold, "aaaa0004", lambda sd, sub: 0.97 if sd != 2 or sub.split(".")[1] == "rot90" and "rot90.rot90" not in sub else 0.6)
    register(lm, ds_gold, "aaaa0005", lambda sd, sub: 0.85)                         # 0.85^12 = 0.14: between the two cuts

    def multi(key):
        return ds_run.change_keys([key]).split_multi_replies()

    # T1 regression: all levers off == the original production block, for 1- and 2-test tasks
    for key in ("aaaa0001", "aaaa0002", "aaaa0004"):
        d0 = tempfile.mkdtemp()
        sys.stdout = quiet
        try:
            original_block(lm, multi(key), d0)
        finally:
            sys.stdout = real_stdout
        mine, _ = run(lm, multi(key), None)
        assert same(load(d0), mine) and len(mine) > 0, key
    base1, _ = run(lm, multi("aaaa0001"), None)
    assert len(base1) == 16 and gold_in(base1, "aaaa0001") == 16
    assert len(run(lm, multi("aaaa0002"), None)[0]) == 32

    # T2 greedy NLL == DFS NLL for the same beam (the two decoders score identically)
    m1 = multi("aaaa0001").augment(n=2, seed=2)
    sub = sorted(m1.keys)[0]
    toks = [tok.encode(m1.get(sub, fmt)["input"])]
    dfs = ns["inference_turbo_dfs"](lm, toks, fmt.max_new_tokens(), -np.log(0.2), time.time() + 60)
    gre = greedy_decode(lm, toks, fmt.max_new_tokens(), time.time() + 60, ns["_arc_token_ids"], EOS, PAD)
    assert gre[0][1][0][1] == dfs[0][1][0][1] and abs(gre[0][1][0][0] - dfs[0][1][0][0]) < 1e-4
    assert abs(gre[0][1][0][0] + L * math.log(0.97)) < 1e-3

    # T3 G1: zero-candidate input gets greedy grids; inputs with candidates are untouched
    assert len(run(lm, multi("aaaa0003"), None)[0]) == 0
    g1, r = run(lm, multi("aaaa0003"), {"g1": "empty_input"})
    assert len(g1) == 16 and all(k.endswith(".rung") for k in g1) and gold_in(g1, "aaaa0003") == 16
    assert all(s["beam_score"] > -np.log(0.2) for v in g1.values() for s in v) and r["stats"]["g1_inputs"] == 1
    assert same(run(lm, multi("aaaa0001"), {"g1": "empty_input"})[0], base1)
    assert len(run(lm, multi("aaaa0004"), {"g1": "empty_view"})[0]) == 16      # per-view mode fills the 14 empty views

    # T4 G3: 2 agreeing views -> 16 extra views, loaded by the production decoder under the same input
    b4, _ = run(lm, multi("aaaa0004"), None)
    assert len(b4) == 2 and agreement({k: v for k, v in b4.items()}) == 2
    g3, r = run(lm, multi("aaaa0004"), {"g3": True})
    assert sum(k.endswith(".run2") for k in g3) == 16 and r["stats"]["g3_inputs"] == 1
    assert same(run(lm, multi("aaaa0001"), {"g3": True, "g2": True})[0], base1)   # consensus -> no second pass
    d = tempfile.mkdtemp()
    for k, v in g3.items():
        with bz2.BZ2File(os.path.join(d, k), "w") as fh:
            pickle.dump(v, fh)
    dec = arc_decoder.ArcDecoder(types.SimpleNamespace(queries={}, replies={}), n_guesses=2)
    dec.load_decoded_results(d)
    assert list(dec.decoded_results) == ["aaaa0004_0"] and len(dec.decoded_results["aaaa0004_0"]) >= 18
    cands = select_v2.group(dec.decoded_results["aaaa0004_0"])
    top = max(cands, key=lambda c: c["n_views"])
    assert np.array_equal(top["grid"], replies["aaaa0004"][0]) and top["n_views"] == 18 and len(top["geos"]) == 8
    assert np.array_equal(dec.run_selection_algo()["aaaa0004_0"][0], replies["aaaa0004"][0])

    # T5 G2: beams between the two cuts appear only with the loose pass, none duplicated from pass 1
    assert len(run(lm, multi("aaaa0005"), None)[0]) == 0
    g2, r = run(lm, multi("aaaa0005"), {"g2": True})
    assert g2 and all(k.endswith(".run3") for k in g2) and gold_in(g2, "aaaa0005") == len(g2)
    assert all(-np.log(0.2) <= s["beam_score"] < -np.log(LOOSE_P) for v in g2.values() for s in v)
    mixed, _ = run(lm, multi("aaaa0004"), {"g2": True})
    firsts = {k: v for k, v in mixed.items() if not k.endswith(".run3")}
    assert same(firsts, b4)
    assert all(s["beam_score"] >= -np.log(0.2) for k, v in mixed.items() if k.endswith(".run3") for s in v)

    # T6 time gates: an exhausted cap or end_time stops every extra pass (and pass 1, like production)
    assert len(run(lm, multi("aaaa0003"), {"g1": "empty_input", "g3": True, "g2": True}, start=time.time() - 5000)[0]) == 0
    assert len(run(lm, multi("aaaa0003"), {"g1": "empty_input"}, cap_end=-1)[0]) == 0
    now = 1000.0
    assert fair_share_seconds(now + 40000, 116, now=now) == 40000 / 30
    assert fair_share_seconds(now + 4000, 116, now=now) == 600.0 and fair_share_seconds(now + 9e9, 0, now=now) == 2400.0

    # T8 probe_gold: decode output untouched; gold NLL = L * -log(conf) on every view; skipped in a rerun
    gd = tempfile.mkdtemp()
    gp = os.path.join(gd, "sol.json")
    json.dump(replies, open(gp, "w"))

    def probe(key, env=None):
        pd_ = tempfile.mkdtemp()
        d = tempfile.mkdtemp()
        old = os.environ.pop("KAGGLE_IS_COMPETITION_RERUN", None)
        if env:
            os.environ["KAGGLE_IS_COMPETITION_RERUN"] = env
        sys.stdout = quiet
        try:
            decode_task(dict(ctx_for(lm, d), gold_path=gp, probe_dir=pd_), multi(key), time.time(),
                        time.time() + 600, {"probe_gold": True})
        finally:
            sys.stdout = real_stdout
            os.environ.pop("KAGGLE_IS_COMPETITION_RERUN", None)
            if old is not None:
                os.environ["KAGGLE_IS_COMPETITION_RERUN"] = old
        f = os.path.join(pd_, "gold_nll_rank0.jsonl")
        return load(d), ([json.loads(x) for x in open(f)] if os.path.exists(f) else [])

    files, pr = probe("aaaa0001")
    assert same(files, base1) and len(pr) == 1 and pr[0]["generated"] and pr[0]["n_tok"] == L
    assert all(abs(x + L * math.log(0.97)) < 1e-3 for x in pr[0]["gold_aug_nll"]) and len(pr[0]["gold_aug_nll"]) == 8
    files, pr = probe("aaaa0003")                                   # never generated: the probe still measures it
    assert not files and not pr[0]["generated"] and all(abs(x + L * math.log(0.80)) < 1e-3 for x in pr[0]["gold_aug_nll"])
    assert [r["bk"] for r in probe("aaaa0002")[1]] == ["aaaa0002_0", "aaaa0002_1"]
    assert probe("aaaa0001", env="1")[1] == []                      # competition rerun: no probe
    import gen_autopsy
    pd2 = tempfile.mkdtemp()
    json.dump(pr[0], open(os.path.join(pd2, "gold_nll_rank0.jsonl"), "w"))
    assert gen_autopsy.parse_gold_nll(pd2)["aaaa0003_0"]["n_tok"] == L

    # T9 union plumbing: a second model's files carry a tag, load under the same input, and add votes
    d = tempfile.mkdtemp()
    sys.stdout = quiet
    try:
        ra = decode_task(ctx_for(lm, d), multi("aaaa0001"), time.time(), time.time() + 600, None)
        rb = decode_task(dict(ctx_for(lm, d), file_tag=".runm2"), multi("aaaa0001"), time.time(), time.time() + 600, None)
        rc = decode_task(dict(ctx_for(lm, tempfile.mkdtemp()), cap_ceil=700.0), multi("aaaa0003"), time.time(),
                         time.time() + 9e6, {"faircap": True})
    finally:
        sys.stdout = real_stdout
    both = load(d)
    assert len(both) == 32 and sum(k.endswith(".runm2") for k in both) == 16
    assert same({k: v for k, v in both.items() if not k.endswith(".runm2")}, base1)
    assert not unsettled(ra) and not unsettled(rb) and unsettled(rc) and rc["task_cap"] == 700.0 and ra["inputs"] == ["aaaa0001_0"]
    dec = arc_decoder.ArcDecoder(types.SimpleNamespace(queries={}, replies={}), n_guesses=2)
    dec.load_decoded_results(d)
    cands = select_v2.group(dec.decoded_results["aaaa0001_0"])
    assert list(dec.decoded_results) == ["aaaa0001_0"] and max(c["n_views"] for c in cands) == 32
    import queue as _q
    import threading
    q2, done = _q.Queue(), _q.Queue()
    wq = WaitQueue(q2, done, n_workers=2, poll=0.01)
    got = []

    def consume():
        while not wq.empty():
            got.append(wq.get())
    th = threading.Thread(target=consume)
    th.start()
    q2.put("t1")
    done.put(1)
    time.sleep(0.05)
    q2.put("t2")                                                   # arrives while one producer is still running
    time.sleep(0.05)
    assert th.is_alive() and got == ["t1", "t2"] and wq.qsize() == 4
    done.put(1)
    th.join(2)
    assert not th.is_alive() and wq.get() is None

    # T7 decision rule
    base = {"n_inputs": 172, "zero_cand": 0, "zero_by_class": {}, "wrong_only": 90,
            "gold_bins": {"p>=.8": 30, "p .5-.8": 8, "p .3-.5": 1, "p .2-.3": 0},
            "low_consensus_with_gold": 1, "low_consensus_without_gold": 1}
    assert decide(base)["levers"] == LEVERS_OFF and not decide(base)["changed"]
    assert decide(dict(base, zero_cand=20), slack=0.4)["levers"] == dict(LEVERS_OFF, g1="empty_input", faircap=True)
    cens = dict(base, gold_bins={"p>=.8": 10, "p .5-.8": 8, "p .3-.5": 6, "p .2-.3": 4})
    assert decide(cens, slack=0.4)["levers"]["g2"] and not decide(cens, slack=0.05)["levers"]["g2"]
    weak = dict(base, low_consensus_with_gold=4, low_consensus_without_gold=6)
    assert decide(weak)["levers"]["g3"] and not decide(weak)["levers"]["faircap"]
    assert not decide(weak, slack=0.1)["levers"]["g3"]

    print(f"selftest ok (torch {torch.__version__}, fake LM, {lm.calls} forward calls): levers-off output identical to the "
          "production decode block on 3 tasks; greedy NLL == DFS NLL; G1/G3/G2 produce the expected files, "
          "leave settled inputs untouched and respect the time gates; probe_gold reports the exact gold NLL "
          "without changing any output; a tagged second-model pass unions under the same input; WaitQueue "
          "waits for late producers; decide() rules hold")


if __name__ == "__main__":
    if len(sys.argv) >= 2 and sys.argv[1] == "--selftest":
        if torch is None:
            sys.exit("selftest needs torch (CPU is enough)")
        selftest()
    elif len(sys.argv) >= 3 and sys.argv[1] == "--decide":
        diag = json.load(open(sys.argv[2]))
        diag = diag.get("diagnostics", diag)
        slack = float(sys.argv[sys.argv.index("--slack") + 1]) if "--slack" in sys.argv else None
        d = decide(diag, slack)
        print(json.dumps(d["levers"]))
        for line in d["why"]:
            print(" -", line)
    else:
        sys.exit(__doc__)


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

GEN_LEVERS = {'g1': 'off', 'g3': False, 'g2': False, 'faircap': False, 'probe_gold': True}


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/takumuhata/qwen3-4b-grids15-sft-a/transformers/bf16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/working/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        import gen_boost
        gen_boost.decode_task(
            dict(model=model, tokenizer=tokenizer, formatter=formatter, ArcDataset=ArcDataset,
                 inference_turbo_dfs=inference_turbo_dfs, calc_scores=calc_scores, arc_token_ids=_arc_token_ids,
                 EOS_ID=EOS_ID, PAD_ID=PAD_ID, max_new_tokens=max_new_tokens, max_score=max_score,
                 max_seq_length=max_seq_length, dir_outputs=dir_outputs, rank=rank, key=key),
            puzzle_ds_multi, start_time, end_time, levers=GEN_LEVERS, tasks_left=max(queue.qsize() - 4, 0))

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ['0934a4d8', '16b78196', '20270e3b', '269e22fb', '28a6681f', '3dc255db', '4a21e3da', '4c7dc4dd', '53fb4810', '5545f144', '581f7754', '58f5dbd5', '5961cc34', '64efde09', '67e490f4', '6ffbe589', '78332cb0', '7b3084d4', '88bcf3b4', '88e364bc', '9385bd28', '981571dc', 'a251c730', 'a395ee82']:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/working/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo())

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))